# Final evaluation, model export & bias audit (Google Colab)

Runs the **leak-free** evaluation of the submitted pipeline (TF-IDF word+char + structured features
-> LinearSVC / 4-model stacking) and exports the serving artifact.

**Before you start**
1. `Runtime -> Change runtime type`: CPU is fine; choose **High-RAM** if offered (the stack needs ~16 GB).
2. Put `train.csv` (and `test.csv`) in Google Drive at `MyDrive/toxic_project/data/`.
3. Push the latest repo to GitHub (the notebook clones it).
4. Do the **dry run first** (Section 2). Then run the full steps in order. Each step saves to Drive,
   so if Colab disconnects you only repeat the step that was running.

Time on free Colab CPU (rough): cv ~1-1.5 h, ablation ~1-1.5 h, holdout ~1 h, export ~1 h.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/toxic_project'          # Drive folder holding data/ and outputs
DATA_DIR = f'{PROJECT}/data'
OUT_DIR  = f'{PROJECT}/results/metrics'                    # metrics (csv/json/md/png)
MODEL_DIR = f'{PROJECT}/model_artifacts'                   # joblib + metadata
REPO_URL = 'https://github.com/Jainparth22/MLtextclassificationpipeline'

import os
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(MODEL_DIR, exist_ok=True)
assert os.path.exists(f'{DATA_DIR}/train.csv'), 'train.csv not found in Drive: ' + DATA_DIR

In [ ]:
%cd /content
!rm -rf repo && git clone -q {REPO_URL} repo
%cd /content/repo
!git log --oneline | head -3
# Colab already ships numpy/pandas/scipy/scikit-learn/matplotlib/seaborn. Do NOT downgrade them:
# the exact versions Colab uses are recorded in model_metadata.json and become the serving pins.
!pip -q install pytest httpx fastapi uvicorn pydantic > /dev/null
import sklearn, sys, numpy, pandas
print('python', sys.version.split()[0], '| sklearn', sklearn.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__)
!python -m pytest -q -W ignore 2>&1 | tail -3

## 1. Dry run on 20,000 rows (about 5-10 min) - confirms everything works before the long runs

In [ ]:
DRY = f'--data-dir {DATA_DIR} --out-dir /content/dry/metrics --model-dir /content/dry/model --sample 20000'
for step in ['cv', 'ablation', 'holdout', 'export', 'report']:
    print('=== ', step)
    !python src/evaluate.py --step {step} {DRY} 2>&1 | grep -v "^\[" | tail -3

## 2. Full runs
Numbers from the dry run are **not** meaningful (small vocab, 20k rows). Everything below is the real thing.

### 2a. Stratified 5-fold CV with features fit *inside* each fold (fixes the validation leak)

In [ ]:
!python src/evaluate.py --step cv --data-dir {DATA_DIR} --out-dir {OUT_DIR}
print(open(f'{OUT_DIR}/cv_summary.md').read())

### 2b. Ablation (3-fold): word only / char only / word+char / +structured / +keep `*!?` in cleaning

In [ ]:
!python src/evaluate.py --step ablation --data-dir {DATA_DIR} --out-dir {OUT_DIR}
print(open(f'{OUT_DIR}/ablation_summary.md').read())

### 2c. Hold-out: stack vs LinearSVC
Per-class metrics, bootstrap 95% CIs, paired difference, confusion matrices, latency, label-3 error sample, fairness audit.

In [ ]:
!python src/evaluate.py --step holdout --data-dir {DATA_DIR} --out-dir {OUT_DIR} --n-jobs 1
print(open(f'{OUT_DIR}/holdout_summary.md').read())

In [ ]:
from IPython.display import Image, display
display(Image(f'{OUT_DIR}/confusion_matrices.png'))
import pandas as pd
pd.set_option('display.max_colwidth', 200)
display(pd.read_csv(f'{OUT_DIR}/error_analysis_label3.csv').head(15))
display(pd.read_csv(f'{OUT_DIR}/fairness_audit.csv').head(15))

### 2d. Export the serving artifact (fits on ALL rows)

In [ ]:
!python src/evaluate.py --step export --data-dir {DATA_DIR} --out-dir {OUT_DIR} --model-dir {MODEL_DIR} --n-jobs 1
!cat {MODEL_DIR}/model_metadata.json
!cat {MODEL_DIR}/requirements-serve.lock.txt

### 2e. Smoke-test the real API against the real artifact

In [ ]:
import os
os.environ['MODEL_PATH'] = f'{MODEL_DIR}/final_stack_pipeline.joblib'
import sys; sys.path.insert(0, 'src')
from fastapi.testclient import TestClient
import app
with TestClient(app.app) as c:
    print(c.get('/info').json().get('sklearn'))
    for t in ['Thanks, great article!', 'you are an idiot and everyone hates you', 'I will find you and kill you']:
        print(t, '->', c.post('/predict', json={'comment': t}).json())

## 3. Collate and download

In [ ]:
!python src/evaluate.py --step report --out-dir {OUT_DIR}
!cd {PROJECT} && zip -rq toxic_results.zip results model_artifacts/model_metadata.json model_artifacts/requirements-serve.lock.txt
from google.colab import files
files.download(f'{PROJECT}/toxic_results.zip')
print('Also keep final_stack_pipeline.joblib in Drive: ', MODEL_DIR)

**Send back** to continue: the contents of `results/metrics/SUMMARY.md`, `holdout.json`, `error_analysis_label3.csv`,
`fairness_audit.csv`, the confusion-matrix image and `model_metadata.json` (the zip has them all).
The model file itself stays in Drive.